In [1]:
import numpy as np
import pandas as pd
import  tpqoa
import time
from datetime import datetime, timedelta, timezone
import warnings


warnings.filterwarnings('ignore')

In [65]:
class ConTrader(tpqoa.tpqoa):
    def __init__(self, conf_file, instrument, bar_length, risk_percentage):
        super().__init__(conf_file)
        self.instrument = instrument
        self.bar_length = pd.Timedelta(bar_length)
        self.tick_data = pd.DataFrame()
        self.raw_data = None
        self.last_bar = None
        self.units = 0
        self.signal = {}  #contains strategy_id, timestamp, symbol, direction, entry_type, stop_loss, take_profit, time_stop, ml_probability, ml_regime_label, confidence_score, valid_until, context_tags,reason_code
        self.position = 0
        self.trade_count = 0
        self.current_trade = 0
        self.trade_created_at = 0
        self.summary = self.get_account_summary()
        self.previous_balance = self.summary['balance']
        self.balance = self.summary['balance']
        self.capital = self.summary['balance']
        self.pl = self.summary['pl']
        self.unit = 0
        self.sl_amount = 0
        self.tp_amount = 0
        self.entry = 0
        self.profit = 0
        self.loss = 0
        self.equity = []
        self.leverage = 100
        self.profit_returns = []
        self.loss_returns = []
        self.profits = [] # NEW

        # unit = amount risk/sl distance
        # amount risk = risk_percentage * balance
        # sl distance = entryprice * 0.005

        #*****************add strategy-specific attributes here******************
        self.obs = []
        self.active_zones = []
        #************************************************************************

    def get_most_recent(self, days = 5):
        now = datetime.now(timezone.utc).replace(tzinfo=None)
        now = now.replace(second=0, microsecond=0)  # floor to the hour
        past = now - timedelta(days = days)
        df = self.get_history(instrument = self.instrument, start = past, end = now,
                               granularity = 'M1', price = "B")
        # df = df.resample(self.bar_length, label = "right").last().dropna().iloc[:-1]

        self.raw_data = df.copy()
        self.dataset_structure()
        for i in range(1, len(self.raw_data)):
            self.analyze_gold_obs(i, take_trade=False)

        self.last_bar = self.raw_data.index[-1]

    def dataset_structure(self):
        self.raw_data['body'] = (self.raw_data['c'] - self.raw_data['o']).abs()
        self.raw_data['ATR_14'] = self.raw_data['c'].rolling(14).mean()
        self.raw_data['time'] = self.raw_data.index
        self.raw_data['atr_norm'] = self.raw_data['ATR_14'] / self.raw_data['ATR_14'].rolling(252).mean()

        self.raw_data['vol_regime'] = pd.qcut(
            self.raw_data['atr_norm'],
            q=[0, 0.33, 0.66, 1.0],
            labels=['Low', 'Medium', 'High'])

        self.raw_data['hour'] = self.raw_data.index.hour

        conditions = [
            self.raw_data['hour'].between(0, 8),
            self.raw_data['hour'].between(8, 9),
            self.raw_data['hour'].between(9, 13),
            self.raw_data['hour'].between(13, 17),
            self.raw_data['hour'].between(17, 22),
            self.raw_data['hour'].between(22, 23)
        ]

        choices = [
            'asian',
            'asian/london',
            'london',
            'london/NY',
            'NY',
            'Closing'
        ]

        self.raw_data['sessions'] = np.select(conditions, choices, default='off')


    def on_success(self, time, bid, ask):
        print(self.ticks, end = " ")

        # collect and store tick data
        recent_tick = pd.to_datetime(time).replace(tzinfo=None)
        df = pd.DataFrame({self.instrument:(ask + bid)/2},
                          index = [recent_tick])
        self.tick_data = pd.concat([self.tick_data, df]) # new with pd.concat()

        # if a time longer than the bar_lenght has elapsed between last full bar and the most recent tick
        if recent_tick - self.last_bar >= self.bar_length:
            self.resample_and_join()
            # self.analyze_gold_obs(len(self.raw_data))

            if self.signal :
                self.execute_trades()

    def resample_and_join(self):
        candle_data = pd.DataFrame({
                        'c': self.tick_data['XAU_USD'].iloc[-1],
                        'o': self.tick_data['XAU_USD'].iloc[0],
                        'h': self.tick_data['XAU_USD'].max(),
                        'l': self.tick_data['XAU_USD'].min(),
                        'hour': self.tick_data.index.hour[-1]
                       }, index=[self.tick_data.index[-1]])

        self.raw_data = pd.concat([self.raw_data, candle_data])
        self.tick_data = self.tick_data.iloc[-1:]
        self.dataset_structure()
        self.last_bar = self.raw_data.index[-1]

    def analyze_gold_obs(self, i, displacement_mult=2.0, forward_window=10, take_trade = True):
        """
        df: DataFrame with ['o', 'h', 'l', 'c', 'ATR_14']
        type: either bullish or bearish
        displacement_mult: How much stronger the move must be than the OB candle to count.
        forward_window: How many candles to look ahead for return after a hit.
        """
        curr = self.raw_data.iloc[i]
        prev = self.raw_data.iloc[i-1]

        if self.current_trade != 1:
            if self.position == 1:
                self.current_trade = 1
                self.execute_trades()

        if curr['c'] > curr['o'] and (curr['c'] - curr['o']) > (prev['h'] - prev['l']) * displacement_mult and \
                prev['body'] < prev['ATR_14']:
            if prev['c'] < prev['o']:
                self.active_zones.append({
                    'type': 'Bullish',
                    'top': prev['h'],
                    'bottom': prev['l'],
                    'created_at': i,
                    'created_time': self.raw_data['time'].iloc[i],
                    'status': 'Active'
                })

        for zone in self.active_zones:
            if zone['status'] != 'Active': continue

            # Check for INVALIDATION (Body Close through zone)
            if zone['type'] == 'Bullish' and curr['c'] < zone['bottom']:
                zone['status'] = 'Invalidated'
                continue
            hit = False
            if zone['type'] == 'Bullish':
                # Low crosses mid of order block
                if curr['l'] <= ((zone['top'] + zone['bottom'])/2) and i - zone['created_at'] > 10:
                    hit = True
            if hit and take_trade:
                self.signal = {
                    'status': 'active',
                    'position': 1,
                    'tp': 100,
                    'sl': 25,
                    'entry_type': "market",
                    'ml_prob':0.6
                }


                self.obs.append({
                    'Type': zone['type'],
                    'Created_At': self.raw_data.index[zone['created_at']],
                    'time': self.raw_data['time'].iloc[i],
                    'vol_regime': self.raw_data['vol_regime'].iloc[i],
                    'sessions': self.raw_data['sessions'].iloc[i],
                    'Hit_At': self.raw_data.index[i],
                    # 'highest_after_hit':self.raw_data['highest'].iloc[i],
                    # 'lowest_after_hit':self.raw_data['lowest'].iloc[i],
                    'Zone_Top': zone['top'],
                    'Zone_Bottom': zone['bottom']
                })
                zone['status'] = 'Mitigated'  # Mark as done

        if i - self.trade_created_at > 20 and self.position == 1:
            print('going neutral')
            self.execute_trades('sell')

            if self.balance > self.previous_balance:
                self.profit += 1

            elif self.balance < self.previous_balance:
                self.loss += 1

            self.previous_balance = self.balance

    def execute_trades(self, position='buy'):
        if position == "sell":
            order = self.create_order(self.instrument, - self.units, suppress = True, ret = True)
            self.report_trade(order, "Closing Long Position")  # NEW
            self.signal['status'] = 'inactive'

        if self.signal:
            if self.signal["position"] == 1:
                if self.position == 0:
                    order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                    self.report_trade(order, "GOING LONG")  # NEW
                    self.signal['status'] = 'inactive'

                elif self.position == -1:
                    order = self.create_order(self.instrument, self.units * 2, suppress = True, ret = True)
                    self.report_trade(order, "GOING LONG")  # NEW
                    self.signal['status'] = 'inactive'

                self.position = 1


            elif self.signal["position"] == -1:
                if self.position == 0:
                    order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                    self.report_trade(order, "GOING SHORT")  # NEW
                    self.signal['status'] = 'inactive'

                elif self.position == 1:
                    order = self.create_order(self.instrument, -self.units * 2, suppress = True, ret = True)
                    self.report_trade(order, "GOING SHORT")  # NEW
                    self.signal['status'] = 'inactive'

                self.position = -1


            elif self.signal["position"] == 0:
                if self.position == -1:
                    order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                    self.report_trade(order, "GOING NEUTRAL")  # NEW
                    self.signal['status'] = 'inactive'

                elif self.position == 1:
                    order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                    self.report_trade(order, "GOING NEUTRAL")  # NEW
                    self.signal['status'] = 'inactive'

                self.position = 0

    def report_trade(self, order, going):  # NEW
        time = order["time"]
        units = order["units"]
        price = order["price"]
        pl = float(order["pl"])
        self.profits.append(pl)
        cumpl = sum(self.profits)
        print("\n" + 100* "-")
        print("{} | {}".format(time, going))
        print("{} | units = {} | price = {} | P&L = {} | Cum P&L = {}".format(time, units, price, pl, cumpl))
        print(100 * "-" + "\n")


In [66]:
trader = ConTrader('../../oanda.cfg', 'XAU_USD', bar_length='5min', risk_percentage=2)

In [67]:
trader.get_most_recent(days=1)

In [74]:
trader.stream_data(trader.instrument)

1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269 270 271 272 273 274 275 276 277 

V20ConnectionError: Connection to v20 REST server at https://stream-fxpractice.oanda.com:443/v3/accounts/101-011-37974904-002/pricing/stream?instruments=XAU_USD&snapshot=True failed

In [73]:
trader.active_zones

[{'type': 'Bullish',
  'top': np.float64(4593.32),
  'bottom': np.float64(4592.57),
  'created_at': 29,
  'created_time': Timestamp('2026-04-28 22:46:00'),
  'status': 'Invalidated'},
 {'type': 'Bullish',
  'top': np.float64(4595.9),
  'bottom': np.float64(4595.35),
  'created_at': 103,
  'created_time': Timestamp('2026-04-29 00:00:00'),
  'status': 'Invalidated'},
 {'type': 'Bullish',
  'top': np.float64(4600.75),
  'bottom': np.float64(4599.7),
  'created_at': 240,
  'created_time': Timestamp('2026-04-29 02:17:00'),
  'status': 'Invalidated'},
 {'type': 'Bullish',
  'top': np.float64(4599.24),
  'bottom': np.float64(4598.1),
  'created_at': 248,
  'created_time': Timestamp('2026-04-29 02:25:00'),
  'status': 'Invalidated'},
 {'type': 'Bullish',
  'top': np.float64(4603.54),
  'bottom': np.float64(4603.12),
  'created_at': 397,
  'created_time': Timestamp('2026-04-29 04:54:00'),
  'status': 'Invalidated'},
 {'type': 'Bullish',
  'top': np.float64(4592.2),
  'bottom': np.float64(4590.3

In [64]:
trader.raw_data

,o,h,l,c,volume,complete,body,ATR_14,time,atr_norm,vol_regime,hour,sessions
time,,,,,,,,,,,,,
2026-04-28 22:04:00,4598.28,4598.28,4597.62,4597.89,20,True,0.39,NaN,2026-04-28 22:04:00,NaN,NaN,22,NY
2026-04-28 22:05:00,4597.90,4598.00,4596.05,4596.18,262,True,1.72,NaN,2026-04-28 22:05:00,NaN,NaN,22,NY
2026-04-28 22:06:00,4596.18,4597.27,4594.86,4595.04,202,True,1.14,NaN,2026-04-28 22:06:00,NaN,NaN,22,NY
2026-04-28 22:07:00,4595.00,4595.56,4593.91,4594.19,287,True,0.81,NaN,2026-04-28 22:07:00,NaN,NaN,22,NY
2026-04-28 22:08:00,4594.19,4595.94,4594.19,4595.70,169,True,1.51,NaN,2026-04-28 22:08:00,NaN,NaN,22,NY
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-29 20:55:00,4542.12,4543.48,4541.82,4543.29,265,True,1.17,4542.792143,2026-04-29 20:55:00,1.000144,High,20,NY
2026-04-29 20:56:00,4543.39,4543.39,4542.30,4542.88,364,True,0.51,4542.700714,2026-04-29 20:56:00,1.000131,High,20,NY
2026-04-29 20:57:00,4542.78,4543.27,4542.70,4543.04,337,True,0.26,4542.633571,2026-04-29 20:57:00,1.000122,Medium,20,NY


In [ ]:
trader.create_order('EUR_USD', 200000, suppress = True, ret = True, sl_distance=0.1, tp_price=1.3)

In [ ]:
trader.get_most_recent()
trader.stream_data(trader.instrument, stop=200)

if trader.position != 0:
    close_order = trader.create_order(trader.instrument, units=-trader.position * trader.units, suppress=True, ret=True)
    trader.report_trade(close_order, "GOING NEUTRAL")
    trader.position = 0

In [28]:
trader.instrument

'XAU_USD'

In [12]:
api = tpqoa.tpqoa('../../oanda.cfg')

In [8]:
api.stream_data("EUR_USD", stop=1)

2026-04-22T23:07:35.877383401Z 1.17051 1.17061
